# BT4012 Competition — Supervised Learning

Continues from `preprocessing.ipynb`. We train on the 31,235 rows with a genuine known label (per that notebook's conclusion, we don't use propagated pseudo-labels) and predict on `test.csv`.

Three models are trained and compared — **Random Forest** (the strongest baseline in the original Elliptic paper, where it beat a GCN), **XGBoost**, and **Logistic Regression** (a non-tree baseline) — using a **temporal** validation split, not a random one: `test.csv` covers `time_step` 36-49, entirely after anything in `train`, so a random split would repeat the same "future leaks into past" mistake found and corrected in `preprocessing.ipynb`. `time_step` itself is excluded as a model input, since every test row's `time_step` is outside the range the model would ever see during training (tree-based models don't extrapolate beyond a feature's observed range).

XGBoost wins the initial comparison and is used for the final submission — but its validation PR-AUC (~0.99) was a poor predictor of the real leaderboard score (0.94), which led to a deeper investigation: **temporal drift** in the illicit-transaction signature, diagnosed with a gap-matched validation scheme and an adversarial-validation check, and addressed with recency-weighted training and tuned hyperparameters (both validated on that gap-matched split rather than the easy adjacent one). Several other hypotheses (stricter feature deduplication, ensembling with Random Forest, new graph-based features) were tested the same way and honestly ruled out.

In [1]:
import itertools

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 20)

## Load data and set up the labeled training set

In [2]:
train_reduced = pd.read_csv("data/train_reduced.csv")
test_reduced = pd.read_csv("data/test_reduced.csv")

# Model features: everything except identifiers/time and the label itself.
feat_cols = [c for c in train_reduced.columns if c not in ("txId", "time_step", "label")]

labeled = train_reduced[train_reduced["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

print("model features:", len(feat_cols))
print("labeled rows:", len(labeled), f"(illicit {labeled['label'].mean():.2%})")
print("test rows:", len(test_reduced))

model features: 76
labeled rows: 31235 (illicit 11.67%)
test rows: 15329


## Temporal validation split

Same split point used to validate label propagation in `preprocessing.ipynb`: train on `time_step` ≤ 30, validate on 31-35. This mirrors the real train → test gap (validation is strictly later than training) so the reported numbers reflect how the model would actually perform on genuinely future data, not an optimistic random split.

In [3]:
SPLIT_POINT = 30

train_df = labeled[labeled["time_step"] <= SPLIT_POINT]
val_df = labeled[labeled["time_step"] > SPLIT_POINT]

X_train, y_train = train_df[feat_cols], train_df["label"]
X_val, y_val = val_df[feat_cols], val_df["label"]

print(f"train: {len(train_df)} rows (illicit {y_train.mean():.2%})")
print(f"val:   {len(val_df)} rows (illicit {y_val.mean():.2%})")

train: 26905 rows (illicit 10.98%)
val:   4330 rows (illicit 15.94%)


## Model 1 — Random Forest

`class_weight="balanced"` re-weights the loss inversely to class frequency, since illicit is only ~11% of labeled rows.

In [4]:
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_val_proba = rf.predict_proba(X_val)[:, 1]
rf_val_pred = rf.predict(X_val)

print(confusion_matrix(y_val, rf_val_pred))
print(classification_report(y_val, rf_val_pred, digits=3))
print("PR-AUC (illicit):", average_precision_score(y_val, rf_val_proba))
print("ROC-AUC:", roc_auc_score(y_val, rf_val_proba))

[[3635    5]
 [ 182  508]]
              precision    recall  f1-score   support

           0      0.952     0.999     0.975      3640
           1      0.990     0.736     0.845       690

    accuracy                          0.957      4330
   macro avg      0.971     0.867     0.910      4330
weighted avg      0.958     0.957     0.954      4330

PR-AUC (illicit): 0.9893704478679826
ROC-AUC: 0.997013855709508


### Random Forest — hyperparameter tuning

`GridSearchCV`'s default `cv` does random/stratified k-fold, which would let training folds contain rows from *after* validation-fold rows in `time_step` — the same "future leaks into past" problem the temporal split above exists to avoid. Instead we use `PredefinedSplit` to force exactly one fold: train on `train_df` (`time_step` ≤ 30), score on `val_df` (`time_step` > 30) — the identical split used to evaluate every other model in this notebook, so results are directly comparable.

In [5]:
# One predefined fold: -1 = never held out (train), 0 = validation fold.
X_gs = pd.concat([X_train, X_val])
y_gs = pd.concat([y_train, y_val])
test_fold = np.concatenate([np.full(len(X_train), -1), np.zeros(len(X_val))])
ps = PredefinedSplit(test_fold)

param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 20],
    "max_features": ["sqrt", "log2"],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": ["balanced", "balanced_subsample"],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=1),
    param_grid=param_grid,
    scoring="average_precision",  # PR-AUC on the illicit class — matches how we judge every other model here
    cv=ps,
    n_jobs=-1,
    refit=False,  # we retrain manually below, on train_df only, to report val metrics without leakage
    verbose=1,
)
grid_search.fit(X_gs, y_gs)

print("candidates tried:", len(grid_search.cv_results_["params"]))
print("best params:", grid_search.best_params_)
print("best PR-AUC (validation fold):", grid_search.best_score_)

Fitting 1 folds for each of 108 candidates, totalling 108 fits
candidates tried: 108
best params: {'class_weight': 'balanced', 'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'n_estimators': 200}
best PR-AUC (validation fold): 0.9897889047761464


In [6]:
rf_tuned = RandomForestClassifier(**grid_search.best_params_, random_state=42, n_jobs=-1)
rf_tuned.fit(X_train, y_train)

rf_tuned_val_proba = rf_tuned.predict_proba(X_val)[:, 1]
rf_tuned_val_pred = rf_tuned.predict(X_val)

print(confusion_matrix(y_val, rf_tuned_val_pred))
print(classification_report(y_val, rf_tuned_val_pred, digits=3))
print("PR-AUC (illicit):", average_precision_score(y_val, rf_tuned_val_proba))
print("ROC-AUC:", roc_auc_score(y_val, rf_tuned_val_proba))

[[3636    4]
 [ 183  507]]
              precision    recall  f1-score   support

           0      0.952     0.999     0.975      3640
           1      0.992     0.735     0.844       690

    accuracy                          0.957      4330
   macro avg      0.972     0.867     0.910      4330
weighted avg      0.958     0.957     0.954      4330

PR-AUC (illicit): 0.9897889047761464
ROC-AUC: 0.9973216276477146


## Model 2 — XGBoost

`scale_pos_weight` is the standard XGBoost equivalent of `class_weight="balanced"` — the ratio of majority to minority class counts in the training fold.

In [7]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, eval_metric="logloss",
    random_state=42, n_jobs=-1,
)
xgb.fit(X_train, y_train)

xgb_val_proba = xgb.predict_proba(X_val)[:, 1]
xgb_val_pred = xgb.predict(X_val)

print(confusion_matrix(y_val, xgb_val_pred))
print(classification_report(y_val, xgb_val_pred, digits=3))
print("PR-AUC (illicit):", average_precision_score(y_val, xgb_val_proba))
print("ROC-AUC:", roc_auc_score(y_val, xgb_val_proba))

[[3614   26]
 [  14  676]]
              precision    recall  f1-score   support

           0      0.996     0.993     0.994      3640
           1      0.963     0.980     0.971       690

    accuracy                          0.991      4330
   macro avg      0.980     0.986     0.983      4330
weighted avg      0.991     0.991     0.991      4330

PR-AUC (illicit): 0.9910822078662335
ROC-AUC: 0.9980303392259914


## Comparison

| Model | Illicit precision | Illicit recall | Illicit F1 | PR-AUC |
|---|---|---|---|---|
| Random Forest | 0.990 | 0.736 | 0.845 | 0.989 |
| **XGBoost** | 0.963 | **0.980** | **0.971** | **0.991** |
| Logistic Regression | — | — | — | 0.52 |

**XGBoost wins clearly**, mainly on recall (0.980 vs. 0.736) — Random Forest is more conservative and misses more illicit transactions, while XGBoost catches nearly all of them at only a small precision cost. Both are dramatically better than the label-propagation approach from `preprocessing.ipynb` (illicit F1 ~0.65 at best there), confirming that a supervised model trained directly on the 76 features vastly outperforms trying to infer labels from neighbor similarity alone. **XGBoost is used for the final submission.**

Logistic Regression (see [Model 3](#Model-3-—-Logistic-Regression-a-non-tree-baseline) below) is far weaker (PR-AUC 0.52, though a fairer tuning raises it closer to ~0.69 — still nowhere near the trees) — the illicit/licit boundary here is genuinely nonlinear. It was included as a diagnostic, not a candidate model, and its results end up feeding directly into the drift investigation below.

## Model 3 — Logistic Regression (a non-tree baseline)

Random Forest and XGBoost are both tree-based — despite bagging vs. boosting being different mechanisms, both ultimately split on feature thresholds, so they can share blind spots (likely part of why ensembling them earlier didn't help: correlated errors, not independent ones).

Logistic Regression is a genuinely different mechanism (a linear decision boundary, not splits), which serves two purposes here:
1. **It directly benefits from the VIF-pruning already done** — VIF exists specifically to fix multicollinearity for linear models, so this feature set is arguably better suited to Logistic Regression than to trees, which don't care about collinearity at all.
2. **It's an independent test of the drift diagnosis.** Every piece of drift evidence so far (label-propagation instability, the adversarial-validation `feat_103` finding, the adjacent-vs-gap-matched PR-AUC drop) comes from tree-based or KNN-based methods. If Logistic Regression *also* shows a large drop from the adjacent-window validation to the gap-matched one, that's strong evidence the drift is a property of the data itself, not an artifact of how trees happen to split feature space.

Features are standardized (zero mean, unit variance) first, since — unlike trees — Logistic Regression's decision boundary and regularization are sensitive to feature scale. The scaler is fit on `X_train` only and applied to `X_val`, to avoid leaking validation-set statistics into training.

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

logreg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)

logreg_val_proba = logreg.predict_proba(X_val_scaled)[:, 1]
logreg_val_pred = logreg.predict(X_val_scaled)

print(confusion_matrix(y_val, logreg_val_pred))
print(classification_report(y_val, logreg_val_pred, digits=3))
print("PR-AUC (illicit):", average_precision_score(y_val, logreg_val_proba))
print("ROC-AUC:", roc_auc_score(y_val, logreg_val_proba))

[[2205 1435]
 [  38  652]]
              precision    recall  f1-score   support

           0      0.983     0.606     0.750      3640
           1      0.312     0.945     0.470       690

    accuracy                          0.660      4330
   macro avg      0.648     0.775     0.610      4330
weighted avg      0.876     0.660     0.705      4330

PR-AUC (illicit): 0.5194696952918673
ROC-AUC: 0.874526198439242


In [9]:
# Interpretability bonus, unavailable from the tree models: which features push toward "illicit" vs. "licit"?
coef_report = pd.Series(logreg.coef_[0], index=feat_cols).sort_values()

print("strongest push toward licit (negative):")
print(coef_report.head(5))
print("\nstrongest push toward illicit (positive):")
print(coef_report.tail(5))

strongest push toward licit (negative):
feat_5    -8.439126
feat_59   -8.062489
feat_22   -6.396555
feat_82   -5.778762
feat_18   -5.700199
dtype: float64

strongest push toward illicit (positive):
feat_83     1.363719
feat_160    1.443927
feat_121    1.562381
feat_108    1.684960
feat_76     1.888138
dtype: float64


## Investigating temporal drift — recency-weighted training

The real test set (`time_step` 36-49) is well beyond the validation window (31-35), and illicit transactions' feature signature is known to drift over `time_step` (see `preprocessing.ipynb`'s label-propagation findings). One cheap mitigation: down-weight older labeled rows during training rather than treating every `time_step` ≤ 30 as equally representative of what the model will face at 36-49.

`recency_weight = 0.5 ** ((SPLIT_POINT - time_step) / half_life)` gives rows at `time_step` == `SPLIT_POINT` a weight of 1.0, decaying by half every `half_life` steps further back. `class_weight="balanced"` (RF) is multiplied automatically with any `sample_weight` passed to `.fit()`; for XGBoost, which only exposes a single global `scale_pos_weight`, we bake the same class ratio directly into a combined per-row weight instead.

This is evaluated on the same `val_df` (31-35) used everywhere else in this notebook, so it's directly comparable to the un-weighted baselines above — but note that's still an *adjacent* window, not a gap matching the real 36-49 distance, so a win here is promising but not the final word.

In [10]:
print("baseline (no recency weight) XGBoost  PR-AUC:", average_precision_score(y_val, xgb_val_proba),
      " ROC-AUC:", roc_auc_score(y_val, xgb_val_proba))
print()

for half_life in [5, 10, 20, 40]:
    recency_weight = 0.5 ** ((SPLIT_POINT - train_df["time_step"]) / half_life)

    class_weight_map = {0: 1.0, 1: scale_pos_weight}
    combined_weight = y_train.map(class_weight_map) * recency_weight

    xgb_rw = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        eval_metric="logloss", random_state=42, n_jobs=-1,
    )
    xgb_rw.fit(X_train, y_train, sample_weight=combined_weight)

    xgb_rw_val_proba = xgb_rw.predict_proba(X_val)[:, 1]
    xgb_rw_val_pred = xgb_rw.predict(X_val)

    print(f"half_life={half_life}")
    print(confusion_matrix(y_val, xgb_rw_val_pred))
    print("PR-AUC:", average_precision_score(y_val, xgb_rw_val_proba),
          " ROC-AUC:", roc_auc_score(y_val, xgb_rw_val_proba))
    print()

baseline (no recency weight) XGBoost  PR-AUC: 0.9910822078662335  ROC-AUC: 0.9980303392259914

half_life=5
[[3609   31]
 [  12  678]]
PR-AUC: 0.9934578302663253  ROC-AUC: 0.9984368530020704

half_life=10
[[3599   41]
 [  14  676]]
PR-AUC: 0.9924189953594335  ROC-AUC: 0.9982843605669692

half_life=20
[[3602   38]
 [  14  676]]
PR-AUC: 0.991756390220275  ROC-AUC: 0.9981247013855709

half_life=40
[[3605   35]
 [  10  680]]
PR-AUC: 0.9937076034550214  ROC-AUC: 0.9986140308966396



### Sanity check: does recency weighting still help at a realistic gap?

The sweep above validated on `time_step` 31-35 with training cut off at 30 — a gap of only 1-5 steps. The real submission trains on all labeled data through 35 and predicts 36-49, a gap of 1-14 steps (average ≈7.5). To check whether the recency-weighting win survives at a comparable distance rather than being an artifact of the easy, adjacent-window validation, we shrink the training cutoff to `time_step` ≤ 20 and predict `time_step` 30-35 — a gap of 10-15 steps, in the same ballpark as the real deployment gap. Fewer training rows is a confound here (we're testing on less data than the ≤30 baseline), but what we care about is the *relative* ranking between the unweighted and recency-weighted models, which this isolates well enough.

In [11]:
GAP_TRAIN_CUTOFF = 20

train_df_gap = labeled[labeled["time_step"] <= GAP_TRAIN_CUTOFF]
val_df_gap = labeled[(labeled["time_step"] >= 30) & (labeled["time_step"] <= 35)]

X_train_gap, y_train_gap = train_df_gap[feat_cols], train_df_gap["label"]
X_val_gap, y_val_gap = val_df_gap[feat_cols], val_df_gap["label"]

scale_pos_weight_gap = (y_train_gap == 0).sum() / (y_train_gap == 1).sum()

print(f"train (gap): {len(train_df_gap)} rows (illicit {y_train_gap.mean():.2%})")
print(f"val   (gap): {len(val_df_gap)} rows (illicit {y_val_gap.mean():.2%})")

xgb_gap_baseline = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_gap, eval_metric="logloss",
    random_state=42, n_jobs=-1,
)
xgb_gap_baseline.fit(X_train_gap, y_train_gap)
xgb_gap_baseline_proba = xgb_gap_baseline.predict_proba(X_val_gap)[:, 1]

print()
print("baseline (no recency weight) XGBoost @ gap  PR-AUC:", average_precision_score(y_val_gap, xgb_gap_baseline_proba),
      " ROC-AUC:", roc_auc_score(y_val_gap, xgb_gap_baseline_proba))
print()

for half_life in [5, 10, 20, 40]:
    recency_weight_gap = 0.5 ** ((GAP_TRAIN_CUTOFF - train_df_gap["time_step"]) / half_life)

    class_weight_map_gap = {0: 1.0, 1: scale_pos_weight_gap}
    combined_weight_gap = y_train_gap.map(class_weight_map_gap) * recency_weight_gap

    xgb_rw_gap = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        eval_metric="logloss", random_state=42, n_jobs=-1,
    )
    xgb_rw_gap.fit(X_train_gap, y_train_gap, sample_weight=combined_weight_gap)

    xgb_rw_gap_proba = xgb_rw_gap.predict_proba(X_val_gap)[:, 1]
    xgb_rw_gap_pred = xgb_rw_gap.predict(X_val_gap)

    print(f"half_life={half_life}")
    print(confusion_matrix(y_val_gap, xgb_rw_gap_pred))
    print("PR-AUC:", average_precision_score(y_val_gap, xgb_rw_gap_proba),
          " ROC-AUC:", roc_auc_score(y_val_gap, xgb_rw_gap_proba))
    print()

train (gap): 18889 rows (illicit 9.38%)
val   (gap): 4854 rows (illicit 15.93%)

baseline (no recency weight) XGBoost @ gap  PR-AUC: 0.821784002649631  ROC-AUC: 0.8835984001841113

half_life=5
[[3999   82]
 [ 182  591]]
PR-AUC: 0.82592456382288  ROC-AUC: 0.8570449053497211

half_life=10
[[3996   85]
 [ 182  591]]
PR-AUC: 0.8370413262065062  ROC-AUC: 0.8866897460956384

half_life=20
[[3993   88]
 [ 187  586]]
PR-AUC: 0.8254250435351971  ROC-AUC: 0.8663766363734633

half_life=40
[[3995   86]
 [ 186  587]]
PR-AUC: 0.8238536539608522  ROC-AUC: 0.8709708607680244



### Does Logistic Regression also show the drift? (independent check)

Same comparison as the XGBoost recency-weighting check above, but for a fundamentally different (linear, non-tree) model — using the identical adjacent-window vs. gap-matched splits already built.

In [12]:
scaler_gap = StandardScaler()
X_train_gap_scaled = scaler_gap.fit_transform(train_df_gap[feat_cols])
X_val_gap_scaled = scaler_gap.transform(val_df_gap[feat_cols])

logreg_gap = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
logreg_gap.fit(X_train_gap_scaled, y_train_gap)
logreg_gap_proba = logreg_gap.predict_proba(X_val_gap_scaled)[:, 1]

print(f"Logistic Regression, adjacent window (31-35)       PR-AUC: {average_precision_score(y_val, logreg_val_proba):.4f}  "
      f"ROC-AUC: {roc_auc_score(y_val, logreg_val_proba):.4f}")
print(f"Logistic Regression, gap-matched (train<=20, 30-35) PR-AUC: {average_precision_score(y_val_gap, logreg_gap_proba):.4f}  "
      f"ROC-AUC: {roc_auc_score(y_val_gap, logreg_gap_proba):.4f}")

Logistic Regression, adjacent window (31-35)       PR-AUC: 0.5195  ROC-AUC: 0.8745
Logistic Regression, gap-matched (train<=20, 30-35) PR-AUC: 0.4386  ROC-AUC: 0.8177


## Which features are actually drift culprits? (adversarial validation)

Rather than lowering the `preprocessing.ipynb` correlation-dedup threshold blindly (which targets feature redundancy, not drift, and could just as easily remove a stable feature and keep a drifting one), we test directly: train a classifier to tell `time_step` ≤ 30 rows apart from `time_step` > 30 rows using **only the 76 features, ignoring `label` entirely**. If it succeeds, its feature importances tell us exactly which columns carry a time signature rather than a genuine illicit-transaction signature — those are the ones worth dropping.

In [13]:
adv_target = (labeled["time_step"] > SPLIT_POINT).astype(int)  # 1 = "late" (time_step > 30), 0 = "early"
X_adv = labeled[feat_cols]

X_adv_train, X_adv_test, y_adv_train, y_adv_test = train_test_split(
    X_adv, adv_target, test_size=0.3, stratify=adv_target, random_state=42
)

adv_clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
adv_clf.fit(X_adv_train, y_adv_train)
adv_auc = roc_auc_score(y_adv_test, adv_clf.predict_proba(X_adv_test)[:, 1])

print(f"adversarial (early vs. late time_step) ROC-AUC: {adv_auc:.4f}")
print("(0.5 = features look identical across time; higher = features carry a time signature)")

adversarial (early vs. late time_step) ROC-AUC: 1.0000
(0.5 = features look identical across time; higher = features carry a time signature)


In [14]:
# Cross-reference drift-importance against label-relevance (computed on train_df only, matching preprocessing.ipynb's convention):
# a feature high in drift-importance but low in label-relevance is close to pure "era" noise — a strong drop candidate.
drift_importance = pd.Series(adv_clf.feature_importances_, index=feat_cols)
label_relevance = train_df[feat_cols].corrwith(train_df["label"]).abs()

drift_report = pd.DataFrame({
    "drift_importance": drift_importance,
    "label_corr": label_relevance,
}).sort_values("drift_importance", ascending=False)

print(drift_report.head(15))

drift_threshold = drift_report["drift_importance"].quantile(0.75)
relevance_threshold = drift_report["label_corr"].median()

drop_candidates = drift_report[
    (drift_report["drift_importance"] >= drift_threshold) & (drift_report["label_corr"] <= relevance_threshold)
].index.tolist()

print(f"\n{len(drop_candidates)} drop candidates (high drift-importance, below-median label relevance):")
print(drop_candidates)

          drift_importance  label_corr
feat_103          0.351671    0.194330
feat_136          0.335547    0.179493
feat_2            0.075314    0.023234
feat_142          0.035390    0.185216
feat_112          0.028374    0.177884
feat_109          0.025886    0.058109
feat_3            0.019601    0.047390
feat_148          0.009893    0.195927
feat_144          0.009705    0.095549
feat_89           0.004926    0.276789
feat_10           0.004499    0.039941
feat_154          0.004449    0.123865
feat_4            0.004301    0.038304
feat_130          0.004279    0.045963
feat_53           0.003873    0.312107

6 drop candidates (high drift-importance, below-median label relevance):
['feat_2', 'feat_10', 'feat_4', 'feat_130', 'feat_16', 'feat_131']


In [15]:
# Does dropping these actually help, at the realistic gap? (half_life=10, the validated recency setting)
recency_weight_gap_hl10 = 0.5 ** ((GAP_TRAIN_CUTOFF - train_df_gap["time_step"]) / 10)
combined_weight_gap_hl10 = y_train_gap.map({0: 1.0, 1: scale_pos_weight_gap}) * recency_weight_gap_hl10

feat_cols_reduced = [c for c in feat_cols if c not in drop_candidates]
print(f"reduced feature count: {len(feat_cols_reduced)} (was {len(feat_cols)})")

xgb_gap_reduced = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    eval_metric="logloss", random_state=42, n_jobs=-1,
)
xgb_gap_reduced.fit(train_df_gap[feat_cols_reduced], y_train_gap, sample_weight=combined_weight_gap_hl10)
xgb_gap_reduced_proba = xgb_gap_reduced.predict_proba(val_df_gap[feat_cols_reduced])[:, 1]

print("gap-matched, half_life=10, all 76 features      PR-AUC: 0.8370  ROC-AUC: 0.8867  (from earlier sweep)")
print(f"gap-matched, half_life=10, {len(feat_cols_reduced)} features (drift dropped)  "
      f"PR-AUC: {average_precision_score(y_val_gap, xgb_gap_reduced_proba):.4f}  "
      f"ROC-AUC: {roc_auc_score(y_val_gap, xgb_gap_reduced_proba):.4f}")

reduced feature count: 70 (was 76)
gap-matched, half_life=10, all 76 features      PR-AUC: 0.8370  ROC-AUC: 0.8867  (from earlier sweep)
gap-matched, half_life=10, 70 features (drift dropped)  PR-AUC: 0.8360  ROC-AUC: 0.8617


## XGBoost — hyperparameter tuning

Every hyperparameter search so far tuned Random Forest (via `GridSearchCV`) or the recency `half_life` — XGBoost's own `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`, and `reg_lambda` have been sitting at their original guessed values (300/6/0.1/none) since the start. We tune them here directly at the **gap-matched split** (not the easy adjacent window — we've seen that split pick the wrong winner twice already), with the validated `half_life=10` recency weighting already baked in via `sample_weight`, so we're tuning on top of the fix that's already proven to work rather than instead of it.

A manual grid (not `GridSearchCV`) is used here rather than wiring up `PredefinedSplit` + `sample_weight` together, which is fragile — the manual loop is more transparent and matches the style of every other sweep in this notebook.

In [16]:
recency_weight_gap_hl10 = 0.5 ** ((GAP_TRAIN_CUTOFF - train_df_gap["time_step"]) / 10)
combined_weight_gap_hl10 = y_train_gap.map({0: 1.0, 1: scale_pos_weight_gap}) * recency_weight_gap_hl10

xgb_param_grid = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
    "reg_lambda": [1, 5],
}
keys = list(xgb_param_grid.keys())
combos = list(itertools.product(*xgb_param_grid.values()))
print(f"{len(combos)} candidates")

results = []
for combo in combos:
    params = dict(zip(keys, combo))
    model = XGBClassifier(
        n_estimators=300, eval_metric="logloss", random_state=42, n_jobs=-1, **params,
    )
    model.fit(train_df_gap[feat_cols], y_train_gap, sample_weight=combined_weight_gap_hl10)
    proba = model.predict_proba(val_df_gap[feat_cols])[:, 1]
    results.append({
        **params,
        "pr_auc": average_precision_score(y_val_gap, proba),
        "roc_auc": roc_auc_score(y_val_gap, proba),
    })

xgb_tuning_results = pd.DataFrame(results).sort_values("pr_auc", ascending=False)
print("\ncurrent default (max_depth=6, lr=0.1, subsample=1.0, colsample=1.0, reg_lambda=1) "
      "gap-matched half_life=10 PR-AUC: 0.8370 (from earlier sweep)\n")
xgb_tuning_results.head(10)

72 candidates

current default (max_depth=6, lr=0.1, subsample=1.0, colsample=1.0, reg_lambda=1) gap-matched half_life=10 PR-AUC: 0.8370 (from earlier sweep)



,max_depth,learning_rate,subsample,colsample_bytree,reg_lambda,pr_auc,roc_auc
13,4,0.10,1.0,0.7,5,0.839128,0.895971
4,4,0.05,1.0,0.7,1,0.838005,0.896020
31,6,0.05,1.0,1.0,5,0.837781,0.904306
22,4,0.20,1.0,1.0,1,0.837315,0.902852
38,6,0.10,1.0,1.0,1,0.837041,0.886690
6,4,0.05,1.0,1.0,1,0.836116,0.892300
5,4,0.05,1.0,0.7,5,0.835176,0.895601
30,6,0.05,1.0,1.0,1,0.835026,0.892694
27,6,0.05,0.7,1.0,5,0.834347,0.896574
12,4,0.10,1.0,0.7,1,0.834298,0.880015


## Final model — retrain on all labeled data, predict on test

The validation split above was only to compare models honestly. For the actual submission we retrain XGBoost on **all** 31,235 labeled rows (time_steps 1-35), not just the ones ≤ 30, since we want to use every known label available before predicting on the genuinely unseen `test.csv` (time_steps 36-49).

We also apply the recency weighting validated above. The gap-matched check (train ≤20, predict 30-35 — a temporal distance similar to this model's real 1-14 step gap to `test.csv`) found `half_life=10` was the best recency setting, clearly ahead of the setting that looked best on the easier adjacent-window validation (`half_life=40`), which barely beat the unweighted baseline once the gap was realistic. We use `half_life=10` here, anchored at `time_step` 35 (the most recent labeled step), for the same reason it helped in that check: it should make the model rely somewhat more on the labeled rows temporally closest to the unseen 36-49 range it has to generalize to.

We also apply the hyperparameters found by the XGBoost tuning above (`max_depth=4, learning_rate=0.1, subsample=1.0, colsample_bytree=0.7, reg_lambda=5`), which improved gap-matched PR-AUC from 0.8370 to 0.8391 — the top candidates there consistently favored shallower, more regularized trees over the original guessed `max_depth=6`, consistent with a model that's less prone to overfitting to any single era's idiosyncrasies.

In [17]:
X_full, y_full = labeled[feat_cols], labeled["label"]
scale_pos_weight_full = (y_full == 0).sum() / (y_full == 1).sum()

FINAL_HALF_LIFE = 10
final_cutoff = labeled["time_step"].max()  # 35 — most recent labeled time_step
recency_weight_full = 0.5 ** ((final_cutoff - labeled["time_step"]) / FINAL_HALF_LIFE)
class_weight_map_full = {0: 1.0, 1: scale_pos_weight_full}
combined_weight_full = y_full.map(class_weight_map_full) * recency_weight_full

final_model = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.1, subsample=1.0,
    colsample_bytree=0.7, reg_lambda=5, eval_metric="logloss",
    random_state=42, n_jobs=-1,
)
final_model.fit(X_full, y_full, sample_weight=combined_weight_full)

test_proba = final_model.predict_proba(test_reduced[feat_cols])[:, 1]

final_rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
)
final_rf.fit(X_full, y_full)
rf_test_proba = final_rf.predict_proba(test_reduced[feat_cols])[:, 1]

print("predicted illicit share on test (XGBoost):", (test_proba >= 0.5).mean())
print("predicted illicit share on test (Random Forest):", (rf_test_proba >= 0.5).mean())

predicted illicit share on test (XGBoost): 0.042859938678322135
predicted illicit share on test (Random Forest): 0.03157414051797247


## Build the submission file

`sample_submission.csv` expects `index` = the 0-based row position in `test.csv`, and `target` = the predicted probability of illicit. We already confirmed `test_reduced.csv` preserves `test.csv`'s exact row order, so a plain positional index lines up correctly.

In [18]:
submission = pd.DataFrame({
    "index": np.arange(len(test_reduced)),
    "target": test_proba,
})
submission.to_csv("data/submission.csv", index=False)

print("saved data/submission.csv:", submission.shape)
submission.head()

saved data/submission.csv: (15329, 2)


,index,target
0,0,0.001652
1,1,0.008080
2,2,0.001382
3,3,0.003654
4,4,0.000053


Also saved: the same XGBoost predictions under an explicit name, since `final_model` above is now trained with **combined class-balance + recency weighting** (`combined_weight_full` — `scale_pos_weight` × `half_life=10` recency decay), not plain `scale_pos_weight` alone.

In [19]:
submission_xgb_recency_weighted = pd.DataFrame({
    "index": np.arange(len(test_reduced)),
    "target": test_proba,
})
submission_xgb_recency_weighted.to_csv("data/submission_xgb_recency_weighted.csv", index=False)

print("saved data/submission_xgb_recency_weighted.csv:", submission_xgb_recency_weighted.shape)
submission_xgb_recency_weighted.head()

saved data/submission_xgb_recency_weighted.csv: (15329, 2)


,index,target
0,0,0.001652
1,1,0.008080
2,2,0.001382
3,3,0.003654
4,4,0.000053


Also saved: the Random Forest submission, for comparison (XGBoost above remains the recommended one — see [Comparison](#Comparison)).

In [20]:
submission_rf = pd.DataFrame({
    "index": np.arange(len(test_reduced)),
    "target": rf_test_proba,
})
submission_rf.to_csv("data/submission_rf.csv", index=False)

print("saved data/submission_rf.csv:", submission_rf.shape)
submission_rf.head()

saved data/submission_rf.csv: (15329, 2)


,index,target
0,0,0.013333
1,1,0.030000
2,2,0.000000
3,3,0.036667
4,4,0.000000


## Alternate export: `half_life=20` recency weighting

For comparison against the `half_life=10` submission above, this retrains `final_model`'s recipe (same tuned hyperparameters) with `half_life=20` (a slower decay — the oldest labeled row retains `0.5 ** (34/20) ≈ 30%` of its weight, vs. `half_life=10`'s ≈9%) on the same full labeled dataset and exports it separately, without overwriting `data/submission.csv`.

In [21]:
HALF_LIFE_20 = 20
recency_weight_hl20 = 0.5 ** ((final_cutoff - labeled["time_step"]) / HALF_LIFE_20)
combined_weight_hl20 = y_full.map(class_weight_map_full) * recency_weight_hl20

final_model_hl20 = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.1, subsample=1.0,
    colsample_bytree=0.7, reg_lambda=5, eval_metric="logloss",
    random_state=42, n_jobs=-1,
)
final_model_hl20.fit(X_full, y_full, sample_weight=combined_weight_hl20)

test_proba_hl20 = final_model_hl20.predict_proba(test_reduced[feat_cols])[:, 1]

submission_xgb_hl20 = pd.DataFrame({
    "index": np.arange(len(test_reduced)),
    "target": test_proba_hl20,
})
submission_xgb_hl20.to_csv("data/submission_xgb_recency_hl20.csv", index=False)

print("predicted illicit share on test (half_life=20):", (test_proba_hl20 >= 0.5).mean())
print("saved data/submission_xgb_recency_hl20.csv:", submission_xgb_hl20.shape)
submission_xgb_hl20.head()

predicted illicit share on test (half_life=20): 0.04403418357361863
saved data/submission_xgb_recency_hl20.csv: (15329, 2)


,index,target
0,0,0.001065
1,1,0.013230
2,2,0.002278
3,3,0.001853
4,4,0.000068
